In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "280_build_silver_fact_estimate_item.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.closed_bids_estimate_item_current"
DIM_PROJECT_TABLE = f"{catalog}.silver.dim_project"
DIM_PROJECT_CLASSIFICATION_TABLE = f"{catalog}.silver.dim_project_classification"
DIM_COUNTY_TABLE = f"{catalog}.silver.dim_county"
TARGET_TABLE = f"{catalog}.silver.fact_estimate_item"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build fact_estimate_item
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH src AS (
        SELECT
              md5(COALESCE(e.base_row_key, '')) AS estimate_item_fact_key
            , dp.project_key
            , e.item_specification_key

            , pc.project_classification_key
            , dc.county_key

            , e.base_row_key
            , e.source_row_id
            , e.source_created_at
            , e.source_updated_at
            , e.source_version
            , e.ingestion_run_id
            , e.ingested_at_utc

            , e.source_name

            , e.project_id
            , e.project_name
            , e.project_number
            , e.state_project_number
            , e.control_section_job_csj
            , e.controlling_project_id_ccsj
            , e.project_type
            , e.project_classification
            , e.project_actual_let_date
            , e.project_estimated_let_date
            , e.project_limits_from
            , e.project_limits_to
            , e.county
            , dc.county_location AS location
            , e.district_division
            , e.highway
            , e.short_description
            , e.federal_project_number

            , e.vendor_name
            , e.vendor_name_standardized
            , e.is_engineer_estimate_row

            , e.bid_submit_sequence_number
            , e.bid_rank_sequence_number
            , e.low_bidder_flag
            , e.dbe_goal_percent
            , e.maximum_number_of_working

            , e.bid_code
            , e.alternative_bid_code
            , e.bid_item_sequence_number
            , e.bid_item_description
            , e.measurement_unit
            , e.bid_item_quantity
            , e.bid_item_unit_price_amount
            , e.bid_total_amount

            , e.specification_code
            , e.specification_description
            , e.effective_specification_description
            , e.spec_book_year

            , e.engineer_s_estimate_unit
            , e.sealed_engineer_s_estimate
            , e.sealed_engineer_s_estimate_1
            , e.a_b_engineer_s_estimate_amount
            , e.engineer_estimate_unit_price
            , e.engineer_project_total

            , e.force_account_amount
            , e.force_account_description
            , e.nbi_number
            , e.utility_id

            , e.business_submission_key
            , e.business_item_key
            , e.record_hash

            , e.estimate_item_rank
            , e.estimate_item_version_count
            , e.estimate_item_has_multiple_versions
            , e.estimate_item_changed_across_versions

            , CASE
                  WHEN e.item_specification_key IS NOT NULL THEN TRUE
                  ELSE FALSE
              END AS is_standard_specification_item

            , CASE
                  WHEN e.item_specification_key IS NULL THEN TRUE
                  ELSE FALSE
              END AS is_unmapped_specification_item

            , CASE
                  WHEN COALESCE(e.bid_code, '') LIKE '9602-%'
                    OR UPPER(COALESCE(e.bid_item_description, '')) LIKE '%PAYMENT ADJUSTMENT%'
                  THEN TRUE
                  ELSE FALSE
              END AS is_payment_adjustment_item

            , CASE
                  WHEN COALESCE(e.bid_code, '') LIKE '9606-%'
                    OR UPPER(COALESCE(e.bid_item_description, '')) LIKE '%SPECIAL DEDUCTION%'
                  THEN TRUE
                  ELSE FALSE
              END AS is_special_deduction_item

            , CASE
                  WHEN e.specification_code IS NULL
                   AND e.specification_description IS NULL
                   AND e.effective_specification_description IS NULL
                   AND e.measurement_unit IS NULL
                   AND e.bid_code IS NULL
                   AND e.bid_item_sequence_number IS NULL
                   AND e.engineer_project_total IS NOT NULL
                  THEN TRUE
                  ELSE FALSE
              END AS is_project_level_estimate_header_row

            , CASE
                  WHEN e.item_specification_key IS NULL
                   AND (
                        COALESCE(e.bid_code, '') LIKE '9602-%'
                        OR UPPER(COALESCE(e.bid_item_description, '')) LIKE '%PAYMENT ADJUSTMENT%'
                        OR COALESCE(e.bid_code, '') LIKE '9606-%'
                        OR UPPER(COALESCE(e.bid_item_description, '')) LIKE '%SPECIAL DEDUCTION%'
                        OR (
                            e.specification_code IS NULL
                            AND e.specification_description IS NULL
                            AND e.effective_specification_description IS NULL
                            AND e.measurement_unit IS NULL
                        )
                   )
                  THEN TRUE
                  ELSE FALSE
              END AS is_nonstandard_adjustment_item

            , CASE
                  WHEN e.bid_item_quantity IS NOT NULL
                   AND e.engineer_estimate_unit_price IS NOT NULL
                  THEN e.bid_item_quantity * e.engineer_estimate_unit_price
                  ELSE NULL
              END AS extended_estimate_item_amount_calc

            , CASE
                  WHEN e.bid_item_quantity IS NOT NULL
                   AND e.engineer_estimate_unit_price IS NOT NULL
                   AND e.engineer_project_total IS NOT NULL
                  THEN ABS(
                       e.engineer_project_total
                       - (e.bid_item_quantity * e.engineer_estimate_unit_price)
                  )
                  ELSE NULL
              END AS project_total_vs_extended_estimate_abs_diff

        FROM {SOURCE_TABLE} e
        LEFT JOIN {DIM_PROJECT_TABLE} dp
            ON e.project_id = dp.project_id
        LEFT JOIN {DIM_PROJECT_CLASSIFICATION_TABLE} pc
            ON (
                CASE
                    WHEN COALESCE(e.project_classification, '') = '' THEN 'UNKNOWN'
                    ELSE e.project_classification
                END
            ) = pc.project_classification
        LEFT JOIN {DIM_COUNTY_TABLE} dc
            ON (
                CASE
                    WHEN COALESCE(e.county, '') = '' THEN 'UNKNOWN'
                    WHEN e.county = 'De Witt' THEN 'DeWitt'
                    ELSE e.county
                END
            ) = dc.county
    )

    SELECT
          estimate_item_fact_key
        , project_key
        , item_specification_key

        , project_classification_key
        , county_key

        , base_row_key
        , source_row_id
        , source_created_at
        , source_updated_at
        , source_version
        , ingestion_run_id
        , ingested_at_utc

        , source_name

        , project_id
        , project_name
        , project_number
        , state_project_number
        , control_section_job_csj
        , controlling_project_id_ccsj
        , project_type
        , project_classification
        , project_actual_let_date
        , project_estimated_let_date
        , project_limits_from
        , project_limits_to
        , county
        , location
        , district_division
        , highway
        , short_description
        , federal_project_number

        , vendor_name
        , vendor_name_standardized
        , is_engineer_estimate_row

        , bid_submit_sequence_number
        , bid_rank_sequence_number
        , low_bidder_flag
        , dbe_goal_percent
        , maximum_number_of_working

        , bid_code
        , alternative_bid_code
        , bid_item_sequence_number
        , bid_item_description
        , measurement_unit
        , bid_item_quantity
        , bid_item_unit_price_amount
        , bid_total_amount

        , specification_code
        , specification_description
        , effective_specification_description
        , spec_book_year

        , engineer_s_estimate_unit
        , sealed_engineer_s_estimate
        , sealed_engineer_s_estimate_1
        , a_b_engineer_s_estimate_amount
        , engineer_estimate_unit_price
        , engineer_project_total

        , force_account_amount
        , force_account_description
        , nbi_number
        , utility_id

        , business_submission_key
        , business_item_key
        , record_hash

        , estimate_item_rank
        , estimate_item_version_count
        , estimate_item_has_multiple_versions
        , estimate_item_changed_across_versions

        , is_standard_specification_item
        , is_unmapped_specification_item
        , is_payment_adjustment_item
        , is_special_deduction_item
        , is_project_level_estimate_header_row
        , is_nonstandard_adjustment_item

        , extended_estimate_item_amount_calc
        , project_total_vs_extended_estimate_abs_diff
    FROM src
    WHERE is_project_level_estimate_header_row = FALSE
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Item-level engineer estimate fact table built from silver.closed_bids_estimate_item_current and linked to project, classification, and county dimensions.'
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_TABLE}").collect()[0]["row_count"]

    missing_project_classification_keys = spark.sql(f"""
        SELECT COUNT(*) AS missing_count
        FROM {TARGET_TABLE}
        WHERE project_classification_key IS NULL
    """).collect()[0]["missing_count"]

    missing_county_keys = spark.sql(f"""
        SELECT COUNT(*) AS missing_count
        FROM {TARGET_TABLE}
        WHERE county_key IS NULL
    """).collect()[0]["missing_count"]

    summary = spark.sql(f"""
        SELECT
              COUNT(*) AS fact_row_count
            , SUM(CASE WHEN item_specification_key IS NULL THEN 1 ELSE 0 END) AS null_item_specification_key_count
            , SUM(CASE WHEN is_nonstandard_adjustment_item THEN 1 ELSE 0 END) AS nonstandard_adjustment_item_count
            , SUM(CASE WHEN estimate_item_changed_across_versions THEN 1 ELSE 0 END) AS changed_estimate_item_count
        FROM {TARGET_TABLE}
    """).collect()[0]

    log_run(
        "SUCCESS",
        row_count,
        f"Built {TARGET_TABLE} successfully."
    )

    print("=" * 70)
    print(f"Built {TARGET_TABLE}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)
    print("Summary:")
    print(f"  fact_row_count: {summary['fact_row_count']:,}")
    print(f"  null_item_specification_key_count: {summary['null_item_specification_key_count']:,}")
    print(f"  nonstandard_adjustment_item_count: {summary['nonstandard_adjustment_item_count']:,}")
    print(f"  changed_estimate_item_count: {summary['changed_estimate_item_count']:,}")
    print(f"  missing_project_classification_keys: {missing_project_classification_keys:,}")
    print(f"  missing_county_keys: {missing_county_keys:,}")

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise